# Full load pipeline: Bronze → Silver for CRM customer info
##
# What we do here:
##   1. Read raw data from Bronze table
##   2. Clean & standardize:
##      - Trim spaces
##      - Rename columns
##      - Fix data types
##      - Unify gender & marital status values
##   3. Validate business key (customer_key):
##      - Not null / not empty
##      - Correct format (prefix + length + numbers)
##      - Keep only best row if duplicates
##      - Final check: no nulls or duplicates left
##   4. Remove technical columns (source, file, timestamp)
##   5. Overwrite Silver table with clean data
##
# Why:
##   - Turn messy raw data into trusted, clean customer master data
##   - Catch bad data early → send to quarantine
##   - Guarantee unique customers in Silver for analytics & models

In [0]:



from core.config import (
    BRONZE_TABLE,
    SILVER_TABLE,
    QUARANTINE_TABLE,
    ALLOWED_PREFIX,
    EXPECTED_LENGTH,
    RENAME_MAP
)

from core.transformation import (
    trim_string_columns,
    rename_columns,
    cast_columns,
    standardize_categorical
)

from core.validation import (
    validate_not_null_business_key,
    validate_business_key_format,
    resolve_duplicate_business_keys,
    final_business_key_validation
)

from core.writer import (
    drop_metadata_columns,
    write_full_overwrite
)


def main():

    # 1. Read Bronze
    df = spark.table(BRONZE_TABLE)

    # 2. Transform
    df = trim_string_columns(df)
    df = rename_columns(df, RENAME_MAP)
    df = cast_columns(df)
    df = standardize_categorical(df)

    # 3. Business Key Validations
    df = validate_not_null_business_key(
        df,
        key_col="customer_key",
        quarantine_table=QUARANTINE_TABLE
    )

    df = validate_business_key_format(
        df,
        key_col="customer_key",
        allowed_prefix=ALLOWED_PREFIX,
        expected_length=EXPECTED_LENGTH,
        quarantine_table=QUARANTINE_TABLE
    )

    df = resolve_duplicate_business_keys(
        df,
        key_col="customer_key",
        important_cols=[
            "first_name",
            "last_name",
            "marital_status",
            "gender"
        ],
        quarantine_table=QUARANTINE_TABLE
    )

    df = final_business_key_validation(
        df,
        key_col="customer_key"
    )

    # 4. Drop metadata columns
    df = drop_metadata_columns(
        df,
        metadata_cols=[
            "source",
            "source_file",
            "ingest_timestamp"
        ]
    )

    # 5. Write to Silver (FULL LOAD → overwrite)
    write_full_overwrite(
        df,
        target_table=SILVER_TABLE
    )


if __name__ == "__main__":
    main()